In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from zipfile import ZipFile
from google.colab import files

# 원본 CSV 3개 업로드
uploaded = files.upload()

# 파일명 뒤에 (1) 등이 붙어 있어도 찾을 수 있도록 처리
file_patterns = {
    "방문자": "부산_구군별_월별_방문자_파생변수",
    "거주인구": "행정동별거주인구수",
    "관광체류강도": "관광수요강도_체류강도_raw",
}

raw = {}
source_files = {}

for dataset, pattern in file_patterns.items():
    matches = [
        name for name in uploaded
        if pattern in name and name.lower().endswith(".csv")
    ]

    assert len(matches) == 1, (
        f"{dataset}: 해당 원본 CSV를 한 개만 선택하세요. "
        f"현재 찾은 파일: {matches}"
    )

    filename = matches[0]
    source_files[dataset] = filename

    raw[dataset] = pd.read_csv(
        io.BytesIO(uploaded[filename]),
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False
    )

# 필수 열 확인
required_columns = {
    "방문자": [
        "기준월", "시군구코드", "구군",
        "외지인_일평균", "외국인_일평균",
        "외지인_전월대비_변화율",
        "외국인_전월대비_변화율",
        "외지인_직전3개월평균대비_변화율",
        "외국인_직전3개월평균대비_변화율"
    ],
    "거주인구": [
        "연월", "행정구역코드", "구군", "읍면동",
        "거주자 인구수", "남자 인구수", "여자 인구수"
    ],
    "관광체류강도": [
        "baseYm", "signguCd", "signguNm",
        "tarSjrnDsIxCd", "tarSjrnDsIxVal"
    ]
}

for dataset, columns in required_columns.items():
    missing = set(columns) - set(raw[dataset].columns)
    assert not missing, f"{dataset} 필수 열 누락: {missing}"

# 읽은 자료 확인
summary = pd.DataFrame([
    {
        "데이터": dataset,
        "원본 파일": source_files[dataset],
        "행 수": len(df),
        "열 수": len(df.columns)
    }
    for dataset, df in raw.items()
])

display(summary)

# 시험 시점과 정답 기간을 명확히 기록
TEST_PREDICTION_DATE = pd.Timestamp("2026-02-01 00:00:00")
TEST_LABEL_END_EXCLUSIVE = pd.Timestamp("2026-08-01")

print("시험 예측시점:", TEST_PREDICTION_DATE)
print("시험 정답 기간: 2026-02-01 이상 ~ 2026-08-01 미만")
print("원본 자료 읽기 완료 — 원본 값은 수정하지 않았습니다.")

In [ ]:
# =========================================================
# 1. 공통 설정·검수 함수
# =========================================================
FEATURE_START = "202301"
FEATURE_END = "202602"

audit_rows = []

def record_check(dataset, item, count, note=""):
    audit_rows.append({
        "데이터": dataset,
        "점검항목": item,
        "문제건수": int(count),
        "판정": "통과" if count == 0 else "확인필요",
        "설명": note
    })

def inspect_source(df, dataset, month_col, region_col, number_cols):
    """원본은 유지하고 분석용 복사본을 검사합니다."""
    result = df.copy()

    blank_count = int(
        result.apply(lambda s: s.str.strip().eq("").sum()).sum()
    )
    record_check(dataset, "빈칸 셀", blank_count)

    parsed_month = pd.to_datetime(
        result[month_col], format="%Y%m", errors="coerce"
    )
    record_check(
        dataset, "날짜 변환 실패", parsed_month.isna().sum()
    )

    duplicate = result.duplicated(
        [region_col, month_col], keep=False
    )
    record_check(dataset, "지역·월 중복 행", duplicate.sum())

    # 지역별 최초~최종 월 사이에 빠진 달 검사
    missing_months = 0
    if parsed_month.notna().all():
        for _, group in result.groupby(region_col):
            observed = pd.PeriodIndex(
                pd.to_datetime(group[month_col], format="%Y%m"),
                freq="M"
            )
            expected = pd.period_range(
                observed.min(), observed.max(), freq="M"
            )
            missing_months += len(expected.difference(observed))

    record_check(dataset, "지역별 중간 월 누락", missing_months)

    for col in number_cols:
        result[col] = pd.to_numeric(result[col], errors="coerce")
        record_check(
            dataset, f"{col}: 숫자 변환 실패",
            result[col].isna().sum()
        )
        record_check(
            dataset, f"{col}: 무한대",
            np.isinf(result[col]).sum()
        )

    return result.sort_values(
        [region_col, month_col]
    ).reset_index(drop=True)

def add_prediction_month(observed_series, lag):
    return (
        pd.to_datetime(observed_series, format="%Y%m")
        .dt.to_period("M") + lag
    ).dt.strftime("%Y%m")

def select_feature_period(df):
    return df.loc[
        df["예측기준월"].between(FEATURE_START, FEATURE_END)
    ].copy()


# =========================================================
# 2. 방문자: 기존 변수 검산 + 전월 연결 후보
# =========================================================
visitor_value_cols = [
    "외지인_일평균", "외국인_일평균",
    "외지인_전월대비_변화율",
    "외국인_전월대비_변화율",
    "외지인_직전3개월평균대비_변화율",
    "외국인_직전3개월평균대비_변화율"
]

v = inspect_source(
    raw["방문자"], "방문자",
    "기준월", "시군구코드", visitor_value_cols
)

formula_rows = []

for person in ["외지인", "외국인"]:
    amount = f"{person}_일평균"

    record_check(
        "방문자", f"{amount}: 음수",
        (v[amount] < 0).sum()
    )

    grouped = v.groupby("시군구코드")[amount]
    denominators = {
        f"{person}_전월대비_변화율": grouped.shift(1),
        f"{person}_직전3개월평균대비_변화율":
            grouped.transform(
                lambda s: s.shift(1).rolling(
                    3, min_periods=3
                ).mean()
            )
    }

    for col, denominator in denominators.items():
        recalculated = v[amount] / denominator.where(
            denominator > 0
        ) - 1

        comparable = recalculated.notna()
        difference = (v[col] - recalculated).abs()
        mismatch_count = int(
            (comparable & difference.gt(1e-8)).sum()
        )

        record_check("방문자", f"{col}: 계산 불일치", mismatch_count)

        formula_rows.append({
            "검산지표": col,
            "비교가능행": int(comparable.sum()),
            "비교불가행": int((~comparable).sum()),
            "불일치행": mismatch_count,
            "최대절대차이": difference[comparable].max()
        })

visitor_formula_checks = pd.DataFrame(formula_rows)

visitor_features = v.rename(
    columns={"기준월": "관측월"}
).copy()

visitor_features["예측기준월"] = add_prediction_month(
    visitor_features["관측월"], lag=1
)
visitor_features["시점적용상태"] = (
    "전월자료 연결 후보·공개시점 미확인"
)

visitor_features = select_feature_period(visitor_features)[
    ["예측기준월", "관측월", "시군구코드", "구군"]
    + visitor_value_cols + ["시점적용상태"]
]


# =========================================================
# 3. 거주인구: 2개월 시차 기준 resident_yoy 생성
# =========================================================
population_cols = [
    "거주자 인구수", "남자 인구수", "여자 인구수", "세대수"
]

r = inspect_source(
    raw["거주인구"], "거주인구",
    "연월", "행정구역코드", population_cols
)

record_check(
    "거주인구", "남녀 합계 불일치",
    (
        r["남자 인구수"] + r["여자 인구수"]
        != r["거주자 인구수"]
    ).sum()
)
record_check(
    "거주인구", "0 이하 인구",
    (r["거주자 인구수"] <= 0).sum()
)

r["월"] = pd.to_datetime(
    r["연월"], format="%Y%m"
).dt.to_period("M")

previous_year = r[
    ["행정구역코드", "월", "거주자 인구수"]
].copy()

previous_year["월"] = previous_year["월"] + 12
previous_year = previous_year.rename(
    columns={"거주자 인구수": "전년동월_거주인구"}
)

resident_features = r.merge(
    previous_year,
    on=["행정구역코드", "월"],
    how="left",
    validate="one_to_one"
)

resident_features["resident_yoy"] = (
    resident_features["거주자 인구수"]
    / resident_features["전년동월_거주인구"].where(
        resident_features["전년동월_거주인구"] > 0
    ) - 1
)

resident_features["관측월"] = (
    resident_features["월"].dt.strftime("%Y%m")
)
resident_features["전년비교월"] = (
    resident_features["월"] - 12
).dt.strftime("%Y%m")
resident_features["예측기준월"] = (
    resident_features["월"] + 2
).dt.strftime("%Y%m")

resident_features = resident_features.rename(columns={
    "행정구역코드": "행정동코드",
    "읍면동": "행정동명",
    "거주자 인구수": "사용월_거주인구"
})

resident_features["시점적용상태"] = (
    "매월1일00시·2개월시차·過去공표일 개별미대조"
)

resident_features = select_feature_period(resident_features)[
    ["예측기준월", "관측월", "전년비교월",
     "행정동코드", "구군", "행정동명",
     "사용월_거주인구", "전년동월_거주인구",
     "resident_yoy", "시점적용상태"]
]

# 독립적인 원자료 조회로 변화율 재검산
population_lookup = r.set_index(
    ["행정구역코드", "연월"]
)["거주자 인구수"].to_dict()

independent_yoy = []
for row in resident_features.itertuples(index=False):
    current = population_lookup[(row.행정동코드, row.관측월)]
    previous = population_lookup[(row.행정동코드, row.전년비교월)]
    independent_yoy.append(current / previous - 1)

record_check(
    "거주인구", "resident_yoy 원자료 재검산 불일치",
    (
        np.abs(
            resident_features["resident_yoy"].to_numpy()
            - np.array(independent_yoy)
        ) > 1e-10
    ).sum()
)


# =========================================================
# 4. 관광체류강도: 종합지수 유지 + 전월 연결 후보
# =========================================================
t_all = raw["관광체류강도"].copy()

record_check(
    "관광체류강도", "종합지표 21 이외 행",
    (~t_all["tarSjrnDsIxCd"].eq("21")).sum()
)

# 부산 전체는 별도 보관, 점포 연결에는 구군 값만 사용
tourism_busan = t_all.loc[
    t_all["tarSjrnDsIxCd"].eq("21")
    & t_all["signguCd"].eq("0")
].copy()

t_source = t_all.loc[
    t_all["tarSjrnDsIxCd"].eq("21")
    & ~t_all["signguCd"].eq("0")
].copy()

t = inspect_source(
    t_source, "관광체류강도",
    "baseYm", "signguCd", ["tarSjrnDsIxVal"]
)

tourism_features = t.rename(columns={
    "baseYm": "관측월",
    "signguCd": "시군구코드",
    "signguNm": "구군",
    "tarSjrnDsIxVal": "관광체류강도"
})

tourism_features["예측기준월"] = add_prediction_month(
    tourism_features["관측월"], lag=1
)
tourism_features["시점적용상태"] = (
    "전월자료 연결 후보·공개시점 미확인"
)

tourism_features = select_feature_period(tourism_features)[
    ["예측기준월", "관측월", "시군구코드",
     "구군", "관광체류강도", "시점적용상태"]
]


# =========================================================
# 5. 세 후보표의 최종 구조·날짜 검사
# =========================================================
feature_tables = {
    "방문자": visitor_features,
    "거주인구": resident_features,
    "관광체류강도": tourism_features
}

model_value_cols = {
    "방문자": visitor_value_cols,
    "거주인구": ["resident_yoy"],
    "관광체류강도": ["관광체류강도"]
}

summary_rows = []

for name, table in feature_tables.items():
    region_col = (
        "행정동코드" if name == "거주인구" else "시군구코드"
    )

    table = table.sort_values(
        ["예측기준월", region_col]
    ).reset_index(drop=True)
    feature_tables[name] = table

    record_check(
        name, "후보표 연결 키 중복",
        table.duplicated(
            ["예측기준월", region_col], keep=False
        ).sum()
    )
    record_check(
        name, "후보표 지표 결측",
        table[model_value_cols[name]].isna().sum().sum()
    )
    record_check(
        name, "관측월이 예측기준월 이상인 행",
        (table["관측월"] >= table["예측기준월"]).sum()
    )

    # 가정한 시차와 실제 날짜가 같은지 검사
    lag = 2 if name == "거주인구" else 1
    expected_prediction = add_prediction_month(
        table["관측월"], lag
    )
    record_check(
        name, "설정 시차와 날짜 불일치",
        (table["예측기준월"] != expected_prediction).sum()
    )

    summary_rows.append({
        "데이터": name,
        "행 수": len(table),
        "열 수": len(table.columns),
        "예측기준월 시작": table["예측기준월"].min(),
        "예측기준월 종료": table["예측기준월"].max(),
        "시험월 202602 행 수":
            int(table["예측기준월"].eq("202602").sum())
    })

visitor_features = feature_tables["방문자"]
resident_features = feature_tables["거주인구"]
tourism_features = feature_tables["관광체류강도"]
audit_table = pd.DataFrame(audit_rows)

print("자료별 생성 결과")
display(pd.DataFrame(summary_rows))

print("검수에서 확인이 필요한 항목")
problems = audit_table.loc[audit_table["판정"].eq("확인필요")]
if problems.empty:
    print("검수한 구조·계산 항목은 모두 통과했습니다.")
else:
    display(problems)

print("시험월 202602의 사용 관측월")
for name, table in feature_tables.items():
    test_rows = table.loc[table["예측기준월"].eq("202602")]
    print(name, ":", sorted(test_rows["관측월"].unique()))

print("※ 공개일 미확인 사항은 계산 검수 통과와 별개입니다.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
import hashlib

# 실행할 때마다 새 폴더 생성 — 이전 결과 보존
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
root = Path(f"/content/이채우_EDA_변수전달_{run_id}")
root.mkdir(parents=True, exist_ok=False)

# 상태 문구의 오타 정리 — 숫자·날짜 값은 변경하지 않음
resident_features["시점적용상태"] = (
    resident_features["시점적용상태"]
    .str.replace("過去", "과거", regex=False)
)
feature_tables["거주인구"] = resident_features

settings = {
    "방문자": {
        "folder": "01_방문자",
        "definition": """
## 변수 정의
- 외국인·외지인 일평균: 해당 관측월의 일평균 방문자 수
- 전월 변화율: 해당 월 일평균 / 전월 일평균 - 1
- 직전 3개월 평균 대비 변화율:
  해당 월 일평균 / 직전 3개월 일평균 값의 산술평균 - 1
- 변화율은 비율로 저장한다. 0.1은 10% 증가이다.
- 기존 일평균 2개·변화율 4개를 보존했다.
- 외국인 방문자는 외국인 주민과 다른 개념이다.

## 시점
- 원자료 기준월은 관측월이다.
- 후보표의 예측기준월은 관측월 다음 달이다.
- 시험 202602에는 202601 관측값이 연결된다.
- 실제 공개일은 미확인이다. 최종 사용 전에 확인해야 한다.
- 공개가 늦다면 사용 관측월을 조정해야 한다.

## 처리
- 기존 변화율을 검산했으며, 재계산 가능한 기록은 모두 일치했다.
- 파일 초반의 변화율은 과거 원자료 부족으로 일부 검산 불가이다.
- 결측 대체·이상치 삭제·표준화는 적용하지 않았다.
""",
        "eda": """
## 기존 전체 원자료 EDA에서 확인한 사항
- 외국인 변화율은 외지인보다 분포 범위가 넓었다.
- 금정구·남구 외지인에서 3월 증가·4월 감소가 여러 해 반복됐다.
- 동래구·연제구 외국인은 2026.06 급증·2026.07 급감했다.
  이 두 달은 이번 입력 후보표 범위 밖이다.
- 변화의 원인은 미확인이다. 원래 값을 유지했다.
- 구군별 방문자 합계를 부산의 고유 방문자 수로 해석하지 않는다.
"""
    },
    "거주인구": {
        "folder": "02_거주인구",
        "definition": """
## 변수 정의
- 실제 사용한 원자료 열은 '거주자 인구수'이다.
- 수정된 resident_yoy:
  t−2월 거주자 인구 / t−14월 거주자 인구 - 1
- 0.02는 전년 동월 대비 2% 증가이다.
- 시험 202602에는 202512 인구와 202412 인구를 비교한다.
- 통신자료의 거주인구와는 별도의 자료이다.

## 지역 기준
- 담당자 확인 결과 녹산동 값에 신호동 인구가 포함돼 있다.
- 카드·통신과 동일한 합산 범위로 사용한다.
- 추가 합산이나 원자료 인구 수정은 하지 않았다.
- 카드·통신과 거주인구의 행정동 코드 집합은 205개로 일치한다.

## 시점
- 예측시점은 매월 1일 00시로 설정했다.
- 공식 공표 규칙은 익월 1일 12시 이후이며,
  주말·공휴일은 다음 평일 공표이다.
- 전월 값이 예측시점에 미공개이므로 두 달 전 값을 사용했다.
- 과거 각 월의 실제 게시일·수정 이력은 개별 대조하지 않았다.
- 공표 규칙 출처: https://jumin.mois.go.kr/statMonth.do

## 명세 변경
- 기존 명세서의 t−1 / t−13 산식에서
  공개 시점을 반영한 t−2 / t−14 산식으로 변경했다.
- 통합 명세서에도 이 변경을 반영해야 한다.
- 사용월_거주인구·전년동월_거주인구는 계산 추적용이다.
""",
        "eda": """
## 기존 원자료·이전 후보표 EDA에서 확인한 사항
- 원자료 기준 부산 거주인구는 전반적인 감소 추세였다.
- 강동동·범일제5동의 큰 증가율을 추가 점검했다.
- 두 동 모두 여러 달에 걸친 인구 증가가 확인됐다.
- 증가 원인은 미확인이며 값을 삭제·대체하지 않았다.
- 이전 후보표의 요약통계는 수정 후보표와 구분해야 한다.
  이번 ZIP의 기초통계·분포 그림은 수정 후보표로 다시 생성했다.
"""
    },
    "관광체류강도": {
        "folder": "03_관광체류강도",
        "definition": """
## 변수 정의
- 종합지표 코드 21의 원래 지수를 사용한다.
- 지수는 방문자 수·백분율이 아니다.
- 100 초과 값을 오류 처리하지 않는다.
- 별도 증감률·표준화·상한 제한은 적용하지 않았다.
- 지수 산식·시기별 산정 방식 변경·소급 개정 여부는 미확인이다.

## 지역 기준
- 부산 전체 행은 원자료에 보존한다.
- 모델 연결 후보표에는 16개 구군 값만 사용한다.
- 같은 구의 점포에는 같은 구군 값이 붙는다.
- 행정동별 실제 관광체류강도로 해석하지 않는다.

## 시점
- 관측월 다음 달을 예측기준월로 지정한 후보표이다.
- 시험 202602에는 202601 지수가 연결된다.
- 실제 공개일은 미확인이다. 최종 사용 전에 확인해야 한다.
- 공개가 늦다면 사용 관측월을 조정해야 한다.
""",
        "eda": """
## 기존 전체 원자료 EDA에서 확인한 사항
- 높은 점수 상위 10건은 모두 해운대구였다.
- 낮은 기록은 북구·사하구에 모였다.
- 여러 구군에서 비슷한 시기의 하락·반등이 관찰됐다.
- 원인은 미확인이다. 원래 값을 유지했다.
- 높은 지수가 낮은 폐업 위험을 뜻한다고 단정하지 않는다.
"""
    }
}

common_note = """
## 공통 적용 기준
- 시험 예측시점: 2026-02-01 00:00
- 시험 정답기간: 2026-02-01 이상, 2026-08-01 미만
- 후보표는 예측기준월 202301~202602 중 확보 가능한 행을 담는다.
- 이 범위의 모든 행을 학습에 사용한다는 뜻은 아니다.
- 학습 정답은 시험 예측시점 전에 관측이 완료돼야 한다.
- 6개월 정답 기준으로 2025.08의 결과는 2026.01 말까지 확인된다.
- 2026.01 기준 점포의 이후 6개월 결과를 학습에 넣으면 안 된다.
- 최종 학습·검증 구간은 통합 학습표에서 확정한다.

## 결합 기준
- 점포 학습표를 기준으로 LEFT JOIN한다.
- 거주인구: 예측기준월 + 행정동코드
- 방문자·관광: 예측기준월 + 시군구코드
- 지역 코드는 문자열로 읽는다.
- 관측월·비교월·시점적용상태는 관리용이며
  모델 예측 변수로 자동 포함하지 않는다.
- 점포 기반 자기 자신 제외·소표본 보정은 이 세 지역 변수에
  일괄 적용하지 않는다.
- 연결 후 점포 행 수·중복·결측·사용 시점을 다시 검수한다.
- 점포 학습표 결합 전이므로 체크리스트 3-1 전체 완료는 아니다.

## 기간별 결측 주의
- 방문자·관광은 202212 원자료가 없어 202301 예측 행이 없다.
- 미래 관측값으로 채우거나 점포를 임의로 삭제하지 않는다.

## 산출물 범위
- 개별 자료 EDA·변수 검산/생성·연결 후보표 준비 완료
- 점포 학습표 결합 및 3-2 통합 EDA는 미진행
- 후보표 공개 시점 관련 한계를 확인 후 최종 채택한다.
"""

created_zips = []

for name, cfg in settings.items():
    folder = root / cfg["folder"]
    graph_dir = folder / "그래프"
    graph_dir.mkdir(parents=True)

    table = feature_tables[name].copy()
    values = model_value_cols[name]

    # 1. 모델 연결 후보 데이터
    table.to_csv(
        folder / f"{name}_모델연결후보.csv",
        index=False, encoding="utf-8-sig"
    )

    # 2. 현재 검수 결과·기초통계
    audit_table.loc[
        audit_table["데이터"].eq(name)
    ].to_csv(
        folder / "검수결과.csv",
        index=False, encoding="utf-8-sig"
    )

    table[values].describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    ).to_csv(
        folder / "후보표_기초통계.csv",
        encoding="utf-8-sig"
    )

    if name == "방문자":
        visitor_formula_checks.to_csv(
            folder / "변화율_검산결과.csv",
            index=False, encoding="utf-8-sig"
        )

    # 3. 출처·변수 정의
    source_filename = source_files[name]
    source_hash = hashlib.sha256(
        bytes(uploaded[source_filename])
    ).hexdigest()

    region_key = (
        "행정동코드" if name == "거주인구" else "시군구코드"
    )

    header = f"""# {name} 변수 정의·처리 기준

- 원본 파일: {source_filename}
- 원본 SHA256: {source_hash}
- 후보표 크기: {len(table):,}행 × {len(table.columns)}열
- 예측기준월: {table['예측기준월'].min()}~{table['예측기준월'].max()}
- 연결 키: 예측기준월 + {region_key}
"""
    (folder / "변수정의_처리기준.md").write_text(
        header + cfg["definition"] + common_note,
        encoding="utf-8"
    )

    # 4. EDA 요약
    current_summary = "\n## 이번 후보표 요약\n"
    for col in values:
        s = table[col]
        current_summary += (
            f"- {col}: 결측 {s.isna().sum()}개, "
            f"최솟값 {s.min():.6g}, "
            f"중앙값 {s.median():.6g}, "
            f"최댓값 {s.max():.6g}\n"
        )

    (folder / "EDA_요약.md").write_text(
        f"# {name} EDA 요약\n"
        + cfg["eda"]
        + current_summary
        + "\n## 그래프 범위\n"
        + "- 아래 그림은 이번 후보표의 지역×월 기록 기준이다.\n"
        + "- 같은 지역이 여러 달 반복되며 독립 표본 수를 뜻하지 않는다.\n"
        + "- 원자료 전체 EDA 기록은 기존 Colab 노트북에 보관한다.\n",
        encoding="utf-8"
    )

    # 5. 후보표 기준 분포 그림
    if name == "방문자":
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        plots = [
            ("외지인_전월대비_변화율", "Domestic visitor MoM"),
            ("외국인_전월대비_변화율", "Foreign visitor MoM")
        ]
        for ax, (col, title) in zip(axes, plots):
            ax.hist(table[col], bins=30, edgecolor="white")
            ax.axvline(0, color="red", linestyle="--")
            ax.set_title(title)
            ax.set_xlabel("Ratio: 0.1 = 10%")
            ax.set_ylabel("District-month records")

    elif name == "거주인구":
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(
            table["resident_yoy"],
            bins=50, edgecolor="white", color="seagreen"
        )
        ax.axvline(0, color="red", linestyle="--")
        ax.set_title("Resident YoY: revised availability lag")
        ax.set_xlabel("Ratio: 0.1 = 10%")
        ax.set_ylabel("Dong-month records")

    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(
            table["관광체류강도"],
            bins=30, edgecolor="white", color="mediumpurple"
        )
        ax.set_title("Tourism stay intensity: feature candidates")
        ax.set_xlabel("Index")
        ax.set_ylabel("District-month records")

    fig.tight_layout()
    fig.savefig(graph_dir / "01_후보표_분포.png", dpi=160)
    plt.close(fig)

    # 6. 지역별 관측값 추이 그림
    # 지역명은 코드로 표시하여 한글 폰트 문제 방지
    trend_col = {
        "방문자": "외국인_일평균",
        "거주인구": "resident_yoy",
        "관광체류강도": "관광체류강도"
    }[name]

    # 거주인구는 205개 선 대신 별도 특이 동 2개만 표시
    if name == "거주인구":
        selected = table.loc[
            table["행정동코드"].isin(
                ["2644053000", "2617061000"]
            )
        ]
        # 코드 추측에 의존하지 않고 이름으로 선택
        selected = table.loc[
            (table["구군"].eq("강서구") & table["행정동명"].eq("강동동"))
            | (table["구군"].eq("동구") & table["행정동명"].eq("범일제5동"))
        ]
        fig, ax = plt.subplots(figsize=(9, 4))
        for code, group in selected.groupby(region_key):
            group = group.sort_values("예측기준월")
            ax.plot(
                pd.to_datetime(group["예측기준월"], format="%Y%m"),
                group[trend_col], label=code
            )
        ax.set_title("Resident YoY: selected dongs")
        ax.set_xlabel("Prediction month")
        ax.set_ylabel("Ratio")
        ax.legend()
        ax.grid(alpha=0.25)

    else:
        fig, axes = plt.subplots(
            4, 4, figsize=(15, 10), sharex=True
        )
        for ax, (code, group) in zip(
            axes.flat, table.groupby(region_key, sort=True)
        ):
            group = group.sort_values("관측월")
            ax.plot(
                pd.to_datetime(group["관측월"], format="%Y%m"),
                group[trend_col]
            )
            ax.set_title(code)
            ax.grid(alpha=0.25)
            ax.tick_params(axis="x", labelrotation=45)
        fig.suptitle(
            "Regional trends: observed month; separate y-axis scales"
        )

    fig.tight_layout()
    fig.savefig(graph_dir / "02_지역별_추이.png", dpi=160)
    plt.close(fig)

    # 그래프 코드·지역명 대응표
    name_col = "행정동명" if name == "거주인구" else "구군"
    mapping_cols = (
        [region_key, "구군", name_col]
        if name == "거주인구"
        else [region_key, name_col]
    )
    table[mapping_cols].drop_duplicates().to_csv(
        folder / "지역코드_이름.csv",
        index=False, encoding="utf-8-sig"
    )

    # 7. 자료별 ZIP 생성
    zip_path = root / f"{cfg['folder']}_EDA_변수전달.zip"
    with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as z:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                z.write(p, arcname=str(p.relative_to(folder)))

    created_zips.append(zip_path)

# 세 ZIP을 한 번에 다운로드하도록 포장
bundle_path = root / "이채우_3개자료_EDA_변수전달.zip"
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as z:
    for p in created_zips:
        z.write(p, arcname=p.name)

print("생성된 자료별 ZIP:")
for p in created_zips:
    print("-", p.name)

print("\n주의: 공개일 미확인 자료는 연결 후보 상태입니다.")
print("점포 학습표 결합 전이므로 3-1 전체 완료는 아닙니다.")

files.download(str(bundle_path))

In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
from google.colab import files

# =====================================================
# 1. 원본 확보
# 기존 raw가 있으면 사용하고, 없으면 거주인구 원본만 업로드
# =====================================================
if "raw" not in globals():
    raw = {}

if "거주인구" not in raw:
    print("행정동별거주인구수_202101_202609.csv를 선택하세요.")
    uploaded_resident = files.upload()

    matches = [
        name for name in uploaded_resident
        if "행정동별거주인구수" in name
        and name.lower().endswith(".csv")
    ]
    assert len(matches) == 1, "거주인구 원본 CSV 한 개를 선택하세요."

    raw["거주인구"] = pd.read_csv(
        io.BytesIO(uploaded_resident[matches[0]]),
        encoding="utf-8-sig",
        dtype=str,
        keep_default_na=False
    )

source = raw["거주인구"].copy()

assert not source.duplicated(
    ["행정구역코드", "연월"]
).any(), "원자료 지역·월 중복 확인 필요"

source["거주자 인구수"] = pd.to_numeric(
    source["거주자 인구수"], errors="raise"
)
source["월"] = pd.to_datetime(
    source["연월"], format="%Y%m", errors="raise"
).dt.to_period("M")

# =====================================================
# 2. 전년 같은 달 인구 연결
# =====================================================
previous = source[
    ["행정구역코드", "월", "거주자 인구수"]
].copy()

previous["월"] = previous["월"] + 12
previous = previous.rename(
    columns={"거주자 인구수": "전년동월_거주인구"}
)

result = source.merge(
    previous,
    on=["행정구역코드", "월"],
    how="left",
    validate="one_to_one"
)

result["resident_yoy"] = (
    result["거주자 인구수"]
    / result["전년동월_거주인구"].where(
        result["전년동월_거주인구"] > 0
    ) - 1
)

# 전월 관측자료를 다음 달 예측에 연결
result["예측기준월"] = (
    result["월"] + 1
).dt.strftime("%Y%m")

result["관측월"] = result["월"].dt.strftime("%Y%m")
result["전년비교월"] = (
    result["월"] - 12
).dt.strftime("%Y%m")

result = result.rename(columns={
    "행정구역코드": "행정동코드",
    "읍면동": "행정동명",
    "거주자 인구수": "사용월_거주인구"
})

result["시점적용상태"] = (
    "소급평가·전월관측자료·실제공개지연 미반영"
)

# 앞서 만든 전달본과 같은 예측기준월 범위 유지
result = result.loc[
    result["예측기준월"].between("202301", "202602"),
    ["예측기준월", "관측월", "전년비교월",
     "행정동코드", "구군", "행정동명",
     "사용월_거주인구", "전년동월_거주인구",
     "resident_yoy", "시점적용상태"]
].sort_values(
    ["예측기준월", "행정동코드"]
).reset_index(drop=True)

# =====================================================
# 3. 원자료에서 독립 재검산
# =====================================================
lookup = source.set_index(
    ["행정구역코드", "연월"]
)["거주자 인구수"].to_dict()

independent = np.array([
    lookup[(r.행정동코드, r.관측월)]
    / lookup[(r.행정동코드, r.전년비교월)] - 1
    for r in result.itertuples(index=False)
])

t = pd.to_datetime(
    result["예측기준월"], format="%Y%m"
).dt.to_period("M")

checks = pd.DataFrame([
    ["연결 키 중복", int(result.duplicated(
        ["예측기준월", "행정동코드"]
    ).sum())],
    ["변화율 결측", int(result["resident_yoy"].isna().sum())],
    ["변화율 무한대", int(np.isinf(result["resident_yoy"]).sum())],
    ["관측월 불일치", int((
        result["관측월"] != (t - 1).dt.strftime("%Y%m")
    ).sum())],
    ["전년비교월 불일치", int((
        result["전년비교월"] != (t - 13).dt.strftime("%Y%m")
    ).sum())],
    ["원자료 재검산 불일치", int((
        np.abs(result["resident_yoy"].to_numpy() - independent)
        > 1e-10
    ).sum())],
], columns=["점검항목", "문제건수"])

checks["판정"] = np.where(
    checks["문제건수"].eq(0), "통과", "확인필요"
)
assert checks["문제건수"].eq(0).all(), "검수 결과를 확인하세요."

# =====================================================
# 4. 새 폴더에 저장 — 이전 결과는 덮어쓰지 않음
# =====================================================
stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
out = Path(f"/content/거주인구_전월기준_{stamp}")
graph_dir = out / "그래프"
graph_dir.mkdir(parents=True, exist_ok=False)

result.to_csv(
    out / "거주인구_모델연결후보.csv",
    index=False, encoding="utf-8-sig"
)
checks.to_csv(
    out / "검수결과.csv",
    index=False, encoding="utf-8-sig"
)
result[["resident_yoy"]].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).to_csv(
    out / "후보표_기초통계.csv",
    encoding="utf-8-sig"
)
result[["행정동코드", "구군", "행정동명"]].drop_duplicates().to_csv(
    out / "지역코드_이름.csv",
    index=False, encoding="utf-8-sig"
)

definition = """# 거주인구 변수 정의·처리 기준 — 전월 기준 복원본

## 사용 기준
- 원자료: 행정동별거주인구수_202101_202609.csv
- 실제 사용 열: 거주자 인구수
- 예측기준월: 202301~202602
- 연결 키: 예측기준월 + 행정동코드
- 행정동코드는 문자열로 읽는다.

## 산식
- resident_yoy = t−1월 인구 / t−13월 인구 - 1
- 예측기준월 202602에는 202601 인구와 202501 인구를 비교한다.
- 0.02는 전년 동월 대비 2% 증가이다.
- 두 달 시차 후보표를 원래 명세서의 전월 기준으로 복원했다.

## 지역 기준
- 담당자 확인 결과 녹산동 인구에는 신호동 인구가 포함돼 있다.
- 추가 합산하지 않았다.
- 카드·통신과 거주인구의 행정동 코드 집합은 205개로 일치한다.

## 평가 가정
- 현재 확보한 과거 월별 자료를 이용한 소급 평가이다.
- 실제 공개 지연과 과거 값의 사후 개정 이력은 반영하지 않았다.
- 실제 월초 운영 성능을 완전히 재현한 결과로 해석하지 않는다.
- 실제 서비스 적용 시 제공 일정에 따른 재검증이 필요하다.

## 모델 결합
- 점포 학습표를 기준으로 LEFT JOIN한다.
- 관측월·전년비교월·시점적용상태는 관리용이다.
- 사용월_거주인구·전년동월_거주인구는 계산 추적용이다.
- 이 관리·추적 열을 모델 변수로 자동 포함하지 않는다.
- 점포 결합 후 행 수·중복·연결률·결측을 다시 검사한다.

## 학습·시험
- 시험 예측기준월: 202602
- 시험 정답기간: 2026-02-01 이상, 2026-08-01 미만
- 후보표의 모든 기준월이 자동으로 학습에 포함되는 것은 아니다.
- 학습·검증 구간과 6개월 정답 관측 가능 여부는 통합 단계에서 확인한다.

## 완료 범위
- 개별 EDA·변수 생성·연결 후보표 준비 완료
- 점포 학습표 결합 및 3-2 통합 EDA는 아직 미진행
"""

(out / "변수정의_처리기준.md").write_text(
    definition, encoding="utf-8"
)

summary = f"""# 거주인구 EDA 요약

- 전월 기준으로 복원한 후보표: {len(result):,}행
- 변화율 결측·연결 키 중복·원자료 재검산 불일치: 모두 0
- 후보표 중앙값: {result['resident_yoy'].median():.2%}
- 후보표 최솟값: {result['resident_yoy'].min():.2%}
- 후보표 최댓값: {result['resident_yoy'].max():.2%}
- 큰 변화율은 삭제하거나 대체하지 않았다.
- 기존 원자료 EDA에서 강동동·범일제5동의 여러 달 인구 증가를 확인했다.
- 증가 원인은 미확인이다.
- 이번 그래프·통계는 전월 기준 복원 후보표로 다시 생성했다.
- 이전 두 달 시차 버전의 통계·그래프와 혼용하지 않는다.
"""
(out / "EDA_요약.md").write_text(summary, encoding="utf-8")

# =====================================================
# 5. 그래프 갱신
# =====================================================
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    result["resident_yoy"],
    bins=50, edgecolor="white", color="seagreen"
)
ax.axvline(0, color="red", linestyle="--")
ax.set_title("Resident YoY: previous-month observations")
ax.set_xlabel("Ratio: 0.1 = 10%")
ax.set_ylabel("Dong-month records")
fig.tight_layout()
fig.savefig(graph_dir / "01_후보표_분포.png", dpi=160)
plt.close(fig)

selected = result.loc[
    (result["구군"].eq("강서구") & result["행정동명"].eq("강동동"))
    | (result["구군"].eq("동구") & result["행정동명"].eq("범일제5동"))
]

fig, ax = plt.subplots(figsize=(9, 4))
for code, group in selected.groupby("행정동코드"):
    ax.plot(
        pd.to_datetime(group["예측기준월"], format="%Y%m"),
        group["resident_yoy"], label=code
    )
ax.set_title("Resident YoY: selected dongs")
ax.set_xlabel("Prediction month")
ax.set_ylabel("Ratio")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(graph_dir / "02_지역별_추이.png", dpi=160)
plt.close(fig)

# =====================================================
# 6. ZIP 생성·다운로드
# =====================================================
zip_path = Path(f"/content/02_거주인구_EDA_전월기준_{stamp}.zip")
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as z:
    for p in sorted(out.rglob("*")):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(out)))

print("행·열 수:", result.shape)
display(checks)

print("202602 시험용 자료")
display(
    result.loc[result["예측기준월"].eq("202602")].head()
)

# 현재 노트북에서도 복원본을 사용하도록 갱신
resident_features = result.copy()
if "feature_tables" in globals():
    feature_tables["거주인구"] = result.copy()

files.download(str(zip_path))

In [ ]:
import io
import pandas as pd
from google.colab import files

# 1. 최신 파일 업로드
license_uploaded = files.upload()

matches = [
    name for name in license_uploaded
    if "인허가_폐업정보결합" in name
    and name.lower().endswith(".csv")
]
assert len(matches) == 1, "최신 인허가 CSV 한 개를 선택하세요."

license_filename = matches[0]

# 2. 원본은 문자열로 보존
license_raw = pd.read_csv(
    io.BytesIO(license_uploaded[license_filename]),
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False
)

required = [
    "점포ID", "점포이력ID",
    "행정동코드", "구군", "업종명", "카드업종",
    "인허가일자_dt", "폐업일자_dt", "영업여부"
]
missing = set(required) - set(license_raw.columns)
assert not missing, f"필수 열 누락: {missing}"

# 3. 작업용 복사본 — 이미 결합된 파일이므로 merge하지 않음
license_joined = license_raw.copy()

license_joined["개업일_변환"] = pd.to_datetime(
    license_joined["인허가일자_dt"].replace("", pd.NA),
    format="%Y-%m-%d",
    errors="coerce"
)

license_joined["폐업일_변환"] = pd.to_datetime(
    license_joined["폐업일자_dt"].replace("", pd.NA),
    format="%Y-%m-%d",
    errors="coerce"
)

# 현재 영업상태는 검수용으로만 구분
license_joined["현재영업여부_검수용"] = (
    license_joined["영업여부"].str.strip()
)

has_close = (
    license_joined["폐업일자_dt"].str.strip().ne("")
)
is_closed = license_joined["현재영업여부_검수용"].eq("F")
is_open = license_joined["현재영업여부_검수용"].eq("T")

# 4. 기본 현황
overview = pd.DataFrame([
    ["원본 행 수", len(license_raw)],
    ["원본 열 수", len(license_raw.columns)],
    ["고유 점포ID 수", license_raw["점포ID"].nunique()],
    ["고유 점포이력ID 수", license_raw["점포이력ID"].nunique()],
    ["행정동 코드 수", license_raw["행정동코드"].nunique()],
    ["업종명 수", license_raw["업종명"].nunique()],
    ["카드업종 수", license_raw["카드업종"].nunique()],
    ["폐업일자가 있는 행 수", int(has_close.sum())],
], columns=["항목", "결과"])

display(overview)

# 5. 오류·불일치 점검
license_checks = pd.DataFrame([
    ["점포ID 빈값",
     int(license_raw["점포ID"].str.strip().eq("").sum())],

    ["점포ID 중복",
     int(license_raw["점포ID"].duplicated().sum())],

    ["점포이력ID 빈값",
     int(license_raw["점포이력ID"].str.strip().eq("").sum())],

    ["행정동코드 빈값",
     int(license_raw["행정동코드"].str.strip().eq("").sum())],

    ["개업일 빈값 또는 변환 실패",
     int(license_joined["개업일_변환"].isna().sum())],

    ["폐업일 값이 있으나 변환 실패",
     int((has_close & license_joined["폐업일_변환"].isna()).sum())],

    ["폐업 상태인데 폐업일 없음",
     int((is_closed & ~has_close).sum())],

    ["영업 상태인데 폐업일 있음",
     int((is_open & has_close).sum())],

    ["T·F 이외 영업상태",
     int((~license_joined["현재영업여부_검수용"].isin(["T", "F"])).sum())],

    ["폐업일이 개업일보다 빠른 행",
     int((
         license_joined["폐업일_변환"]
         < license_joined["개업일_변환"]
     ).sum())],
], columns=["점검 항목", "문제건수"])

license_checks["판정"] = license_checks["문제건수"].map(
    lambda n: "통과" if n == 0 else "확인필요"
)

display(license_checks)

print("현재 영업상태별 기록 수 — 검수용")
display(
    license_joined["현재영업여부_검수용"]
    .value_counts(dropna=False)
    .to_frame("행 수")
)

print("개업일 범위:",
      license_joined["개업일_변환"].min(),
      "~", license_joined["개업일_변환"].max())

print("폐업일 범위:",
      license_joined["폐업일_변환"].min(),
      "~", license_joined["폐업일_변환"].max())

In [ ]:
# 1. 작업용 복사본
work = license_joined.copy()

# 현재 영업여부를 모델 입력이나 정답으로 사용하지 않습니다.
# 앞서 폐업 상태와 폐업일의 일치 여부를 확인했으므로
# 폐업일 존재 건수로 '모두 폐업' 여부를 집계합니다.
grouped = work.groupby("점포이력ID", sort=False)

# 2. 점포이력별 개업·폐업일 집계
master_dates = grouped.agg(
    인허가건수=("점포ID", "size"),
    폐업일있는건수=("폐업일_변환", "count"),
    open_s=("개업일_변환", "min"),
    마지막폐업일=("폐업일_변환", "max")
).reset_index()

master_dates["모두폐업"] = (
    master_dates["인허가건수"]
    == master_dates["폐업일있는건수"]
)

# 하나라도 폐업일이 없는 인허가가 있으면 점포 폐업일은 비워둠
master_dates["close_s"] = master_dates["마지막폐업일"].where(
    master_dates["모두폐업"]
)

# 3. 같은 점포이력 안에서 업종·지역이 다른지 확인
attribute_cols = [
    "업종명", "카드업종", "행정동코드", "구군",
    "프랜차이즈유무"
]

attribute_counts = grouped[attribute_cols].nunique()

history_checks = pd.DataFrame([
    ["고유 점포이력 수", len(master_dates)],
    ["인허가가 2건 이상인 점포이력",
     int(master_dates["인허가건수"].gt(1).sum())],
    ["일부 인허가만 폐업한 점포이력",
     int((
         master_dates["폐업일있는건수"].gt(0)
         & ~master_dates["모두폐업"]
     ).sum())],
    ["모든 인허가가 폐업한 점포이력",
     int(master_dates["모두폐업"].sum())],
    ["업종명이 여러 개인 점포이력",
     int(attribute_counts["업종명"].gt(1).sum())],
    ["카드업종이 여러 개인 점포이력",
     int(attribute_counts["카드업종"].gt(1).sum())],
    ["행정동코드가 여러 개인 점포이력",
     int(attribute_counts["행정동코드"].gt(1).sum())],
    ["구군이 여러 개인 점포이력",
     int(attribute_counts["구군"].gt(1).sum())],
    ["프랜차이즈 여부가 여러 값인 점포이력",
     int(attribute_counts["프랜차이즈유무"].gt(1).sum())],
], columns=["확인 항목", "점포이력 수"])

display(history_checks)

# 4. 날짜 집계 자체 검수
assert master_dates["점포이력ID"].is_unique
assert master_dates["open_s"].notna().all()
assert master_dates["인허가건수"].sum() == len(work)
assert not (
    master_dates["close_s"] < master_dates["open_s"]
).any()

print("점포이력별 날짜 집계 완료:", len(master_dates), "행")
print("원본 인허가 행은 삭제·수정하지 않았습니다.")

# 5. 대표 속성 선택 시 미래 행을 사용할 위험이 있는지 확인
# 명세서의 '최근 인허가 행'을 과거 예측에도 그대로 적용하기 전에 점검
license_date_range = grouped["개업일_변환"].agg(["min", "max"])

test_date = pd.Timestamp("2026-02-01")

crossing_test_date = (
    license_date_range["min"].lt(test_date)
    & license_date_range["max"].ge(test_date)
)

print(
    "시험일 이전·이후 인허가가 함께 있는 점포이력:",
    int(crossing_test_date.sum())
)

# 개인정보 대신 이력ID와 집계 정보만 표시
display(
    master_dates.loc[
        master_dates["인허가건수"].gt(1),
        ["점포이력ID", "인허가건수",
         "폐업일있는건수", "open_s", "close_s"]
    ].head()
)

In [ ]:
# 1. 점포이력별 기본표: 날짜 + 일관된 행정동
history_location = (
    work.groupby("점포이력ID", sort=False)["행정동코드"]
    .first()
    .reset_index()
)

store_master_base = master_dates[
    ["점포이력ID", "open_s", "close_s",
     "인허가건수", "폐업일있는건수", "모두폐업"]
].merge(
    history_location,
    on="점포이력ID",
    how="left",
    validate="one_to_one"
)

assert store_master_base["점포이력ID"].is_unique
assert store_master_base["행정동코드"].str.strip().ne("").all()

print("점포 기본표 생성:", store_master_base.shape)

# 2. 구군이 서로 다른 2개 점포이력의 관련 기록 확인
conflict_ids = attribute_counts.index[
    attribute_counts["구군"].gt(1)
]

print("구군 불일치 기록 — 사업장명·주소는 표시하지 않음")
display(
    work.loc[
        work["점포이력ID"].isin(conflict_ids),
        ["점포이력ID", "점포ID", "행정동코드", "구군",
         "업종명", "카드업종", "인허가일자_dt"]
    ].sort_values(["점포이력ID", "인허가일자_dt"])
)

# 3. 시험시점 이전에 등록된 인허가만 선택
test_date = pd.Timestamp("2026-02-01")
eligible = work.loc[
    work["개업일_변환"].lt(test_date)
].copy()

# 점포이력마다 시험시점 이전의 가장 최근 인허가일
latest_date = eligible.groupby(
    "점포이력ID"
)["개업일_변환"].transform("max")

latest_candidates = eligible.loc[
    eligible["개업일_변환"].eq(latest_date)
].copy()

# 4. 가장 최근 인허가일이 같으면서 속성이 다른 경우 확인
# 날짜가 같으면 '최근 행'이라는 규칙만으로 하나를 고를 수 없습니다.
latest_attribute_counts = latest_candidates.groupby(
    "점포이력ID"
)[["업종명", "카드업종", "프랜차이즈유무"]].nunique()

latest_checks = pd.DataFrame([
    ["시험시점 이전 인허가가 있는 점포이력",
     eligible["점포이력ID"].nunique()],
    ["최근 인허가일 동률·업종명 복수",
     int(latest_attribute_counts["업종명"].gt(1).sum())],
    ["최근 인허가일 동률·카드업종 복수",
     int(latest_attribute_counts["카드업종"].gt(1).sum())],
    ["최근 인허가일 동률·프랜차이즈 여부 복수",
     int(latest_attribute_counts["프랜차이즈유무"].gt(1).sum())],
], columns=["확인 항목", "점포이력 수"])

display(latest_checks)

# 동률 업종 사례는 5개 이력만 확인
tie_ids = latest_attribute_counts.index[
    latest_attribute_counts["업종명"].gt(1)
][:5]

if len(tie_ids):
    print("최근 날짜가 같지만 업종이 다른 사례")
    display(
        latest_candidates.loc[
            latest_candidates["점포이력ID"].isin(tie_ids),
            ["점포이력ID", "점포ID", "인허가일자_dt",
             "업종명", "카드업종", "프랜차이즈유무"]
        ].sort_values(["점포이력ID", "점포ID"])
    )

In [ ]:
# =====================================================
# 1. 행정동코드 → 구군 대응
# 검증한 거주인구 지역표 사용
# =====================================================
region_source = raw["거주인구"][
    ["행정구역코드", "구군"]
].drop_duplicates().rename(
    columns={"행정구역코드": "행정동코드"}
)

assert not region_source["행정동코드"].duplicated().any(), (
    "행정동코드별 구군 대응을 확인하세요."
)

# 점포이력별 날짜·행정동 기본표에 관할 구군 연결
master_location = store_master_base.merge(
    region_source,
    on="행정동코드",
    how="left",
    validate="many_to_one"
)

assert master_location["구군"].notna().all()

# 현재 사용하는 10자리 행정동코드의 앞 5자리: 시군구코드
master_location["시군구코드"] = (
    master_location["행정동코드"].str[:5]
)

# 방문자 자료의 코드·구군 대응과도 일치하는지 확인
district_map = raw["방문자"][
    ["시군구코드", "구군"]
].drop_duplicates()

region_validation = master_location[
    ["시군구코드", "구군"]
].drop_duplicates().merge(
    district_map,
    on=["시군구코드", "구군"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

assert region_validation["_merge"].eq("both").all()

# =====================================================
# 2. 시험일 당시 영업 점포 추출
# 명세서: open_s < t, close_s 없음 또는 close_s >= t
# =====================================================
test_t = pd.Timestamp("2026-02-01")
label_end = test_t + pd.DateOffset(months=6)

test_snapshot = master_location.loc[
    master_location["open_s"].lt(test_t)
    & (
        master_location["close_s"].isna()
        | master_location["close_s"].ge(test_t)
    )
].copy()

test_snapshot["예측기준월"] = "202602"

# 이후 6개월 실제 폐업 여부
test_snapshot["label_close_6m"] = (
    test_snapshot["close_s"].ge(test_t)
    & test_snapshot["close_s"].lt(label_end)
).astype("int8")

# =====================================================
# 3. 시험일 이전의 최근 인허가 속성
# 날짜 동률에서 값이 다르면 해당 속성은 보류
# =====================================================
attribute_group = latest_candidates.groupby("점포이력ID")

representative = attribute_group[
    ["업종명", "카드업종", "프랜차이즈유무"]
].first()

counts = attribute_group[
    ["업종명", "카드업종", "프랜차이즈유무"]
].nunique()

for col in ["업종명", "카드업종", "프랜차이즈유무"]:
    ambiguous = counts[col].gt(1)
    representative[col + "_선택보류"] = ambiguous.astype("int8")

    # 동률인데 값이 다르면 임의의 첫 값을 사용하지 않음
    representative.loc[ambiguous, col] = pd.NA

test_snapshot = test_snapshot.merge(
    representative.reset_index(),
    on="점포이력ID",
    how="left",
    validate="one_to_one"
)

# =====================================================
# 4. 검수·현황
# =====================================================
assert test_snapshot["점포이력ID"].is_unique
assert test_snapshot["행정동코드"].notna().all()
assert test_snapshot["시군구코드"].notna().all()
assert test_snapshot["label_close_6m"].isin([0, 1]).all()

test_summary = pd.DataFrame([
    ["시험 대상 점포 수", len(test_snapshot)],
    ["2~7월 폐업 정답 1",
     int(test_snapshot["label_close_6m"].sum())],
    ["폐업 정답 0",
     int(test_snapshot["label_close_6m"].eq(0).sum())],
    ["시험 대상 중 대표 업종 선택 보류",
     int(test_snapshot["업종명_선택보류"].sum())],
    ["시험 대상 중 카드업종 선택 보류",
     int(test_snapshot["카드업종_선택보류"].sum())],
], columns=["항목", "점포 수"])

display(test_summary)

print("정답 1 비율:",
      f"{test_snapshot['label_close_6m'].mean():.2%}")

print("※ 시험용 점포 표 초안입니다.")
print("※ 업종 동률 규칙과 명세서의 폐업 제외 조건 적용 여부는")
print("   통합 담당자 확인 후 최종 확정해야 합니다.")

In [ ]:
import pandas as pd
import numpy as np

# 원래 시험용 점포 초안은 보존
test_with_features = test_snapshot.copy()

base_count = len(test_with_features)
base_ids = set(test_with_features["점포이력ID"])

# 키 형식 통일
for col in ["예측기준월", "행정동코드", "시군구코드"]:
    test_with_features[col] = (
        test_with_features[col].astype(str).str.strip()
    )

visitor_values = [
    "외지인_일평균",
    "외국인_일평균",
    "외지인_전월대비_변화율",
    "외국인_전월대비_변화율",
    "외지인_직전3개월평균대비_변화율",
    "외국인_직전3개월평균대비_변화율"
]

join_reports = []

def attach_features(base, source, name, region_key, value_cols,
                    extra_cols=None):
    """必要な列だけをLEFT JOINし、接続結果を記録します。"""
    extra_cols = extra_cols or []
    keys = ["예측기준월", region_key]

    cols = keys + value_cols + ["관측월"] + extra_cols
    cols = list(dict.fromkeys(cols))
    right = source[cols].copy()

    for key in keys:
        right[key] = right[key].astype(str).str.strip()

    assert not right.duplicated(keys).any(), (
        f"{name}: 연결 키 중복 확인 필요"
    )

    # 관측월 등 관리 열의 이름 충돌 방지
    rename_cols = {
        col: f"{name}_{col}"
        for col in ["관측월"] + extra_cols
    }
    right = right.rename(columns=rename_cols)

    indicator = f"{name}_연결상태"
    before = len(base)

    joined = base.merge(
        right,
        on=keys,
        how="left",
        validate="many_to_one",
        indicator=indicator,
        sort=False
    )

    observed_col = f"{name}_관측월"
    matched = joined[indicator].eq("both")
    observed = pd.to_datetime(
        joined[observed_col], format="%Y%m", errors="coerce"
    ).dt.to_period("M")
    prediction = pd.to_datetime(
        joined["예측기준월"], format="%Y%m", errors="raise"
    ).dt.to_period("M")

    # 이번 소급 실험의 공통 기준: 전월 관측값
    wrong_month = matched & observed.ne(prediction - 1)

    join_reports.append({
        "자료": name,
        "연결 전 행 수": before,
        "연결 후 행 수": len(joined),
        "연결 성공 행": int(matched.sum()),
        "연결 실패 행": int((~matched).sum()),
        "성공 행 중 지표 결측 셀":
            int(joined.loc[matched, value_cols].isna().sum().sum()),
        "전월 연결 불일치 행": int(wrong_month.sum()),
        "연결률": float(matched.mean())
    })

    assert len(joined) == before, f"{name}: 연결 후 행 수 증가"
    assert not wrong_month.any(), (
        f"{name}: 전월 기준 파일인지 확인하세요."
    )

    return joined

# 1. 거주인구 연결 — 최신 전월 기준 복원본 사용
test_with_features = attach_features(
    test_with_features,
    resident_features,
    name="거주인구",
    region_key="행정동코드",
    value_cols=["resident_yoy"],
    extra_cols=["전년비교월", "시점적용상태"]
)

# 2. 방문자 연결
test_with_features = attach_features(
    test_with_features,
    visitor_features,
    name="방문자",
    region_key="시군구코드",
    value_cols=visitor_values,
    extra_cols=["시점적용상태"]
)

# 3. 관광체류강도 연결
test_with_features = attach_features(
    test_with_features,
    tourism_features,
    name="관광",
    region_key="시군구코드",
    value_cols=["관광체류강도"],
    extra_cols=["시점적용상태"]
)

# 4. 점포와 정답이 그대로인지 확인
assert len(test_with_features) == base_count
assert set(test_with_features["점포이력ID"]) == base_ids
assert not test_with_features.duplicated(
    ["예측기준월", "점포이력ID"]
).any()

before_labels = test_snapshot.set_index(
    "점포이력ID"
)["label_close_6m"].sort_index()

after_labels = test_with_features.set_index(
    "점포이력ID"
)["label_close_6m"].sort_index()

assert before_labels.equals(after_labels), "정답 변경 확인 필요"

# 5. 결과 출력
join_report = pd.DataFrame(join_reports)

display(join_report.style.format({"연결률": "{:.2%}"}))

print("최종 점포 행 수:", len(test_with_features))
print("점포·기준월 중복:",
      test_with_features.duplicated(
          ["예측기준월", "점포이력ID"]
      ).sum())
print("점포 목록·폐업 정답 변경 없음")
print("※ 인허가 최종본이 아닌 연결 검수용 초안입니다.")

# 실패 기록이 있으면 확인용 목록만 표시
for name in ["거주인구", "방문자", "관광"]:
    status = f"{name}_연결상태"
    unmatched = test_with_features.loc[
        test_with_features[status].ne("both"),
        ["예측기준월", "점포이력ID",
         "행정동코드", "시군구코드"]
    ]
    if not unmatched.empty:
        print(f"{name} 미연결 예시")
        display(unmatched.head())

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
from google.colab import files

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
out = Path(f"/content/지역변수_점포연결검수_{stamp}")
out.mkdir(parents=True, exist_ok=False)

# 1. 연결 결과: 최종 학습표가 아닌 시험월 검수용 초안
test_with_features.to_csv(
    out / "202602_시험점포_지역변수연결_검수초안.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

# 2. 연결 검수표
join_report.to_csv(
    out / "지역변수_연결검수표.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. 대표 업종 선택 보류 목록
pending_mask = (
    test_with_features["업종명_선택보류"].eq(1)
    | test_with_features["카드업종_선택보류"].eq(1)
)

test_with_features.loc[
    pending_mask,
    ["예측기준월", "점포이력ID",
     "행정동코드", "시군구코드",
     "업종명", "카드업종",
     "업종명_선택보류", "카드업종_선택보류"]
].to_csv(
    out / "대표업종_選택보류_목록.csv".replace("選", "선"),
    index=False,
    encoding="utf-8-sig"
)

# 4. 팀 전달 설명
note = """# 지역 변수 점포 연결 검수 결과

## 완료 내용
- 시험 예측기준월 202602 점포 초안에 지역 변수 8개를 연결했다.
- 연결 전후 125,468행 유지
- 거주인구·방문자·관광 모두 연결률 100%
- 연결 실패·지표 결측·전월 연결 불일치 0건
- 점포×기준월 중복 0건
- 점포 목록과 폐업 정답 변경 없음

## 연결한 변수
- 거주인구: resident_yoy
- 방문자: 외국인·외지인 일평균 2개, 변화율 4개
- 관광: 관광체류강도 종합지수 1개

## 연결 키
- 거주인구: 예측기준월 + 행정동코드
- 방문자·관광: 예측기준월 + 시군구코드

## 시간 기준
- 전월 관측자료를 사용하는 소급 실험이다.
- 시험 예측기준월 202602에는 202601 관측자료를 연결했다.
- resident_yoy는 202601 인구 / 202501 인구 - 1이다.
- 실제 공개 지연과 과거 자료 개정 이력은 반영하지 않았다.

## 인허가 관련 미확정 사항
- 인허가 최종 검수는 팀원이 진행 중이다.
- 대표 업종 선택 보류 746개, 카드업종 선택 보류 324개
- 두 목록은 중복될 수 있으므로 단순 합산하지 않는다.
- 위 점포도 지역 변수 연결에는 포함했고 삭제하지 않았다.
- 업종 동률 규칙·폐업 제외 조건·점포이력 집계 기준의 최종 확인 필요

## 사용 범위
- 이 파일은 연결 검수용 초안이며 최종 학습표가 아니다.
- 관리용 날짜·연결상태·선택보류 열을 모델 변수로 자동 투입하지 않는다.
- close_s 등 폐업 결과를 담은 열은 입력 변수로 사용하지 않는다.
- 팀의 최종 점포 학습표가 확정되면 동일 연결 절차를 재실행한다.
- 학습 구간 전체 생성·결합 및 3-2 통합 EDA는 아직 남아 있다.
- 시험자료는 연결·정답 품질 점검용으로 사용했으며,
  시험 라벨과의 관계를 이용한 변수 선택은 하지 않았다.
"""

(out / "먼저읽기_팀전달.md").write_text(
    note, encoding="utf-8"
)

# 5. ZIP 다운로드
zip_path = Path(f"/content/지역변수_점포연결검수_{stamp}.zip")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()):
        if p.is_file():
            z.write(p, arcname=p.name)

files.download(str(zip_path))

In [ ]:
print("연결된 점포 표가 준비됐는지:", "test_with_features" in globals())

In [ ]:
# 다음 단계 준비: 점포 기본표 확인
print("점포 기본표 크기:", df.shape)
print("통합점포ID 중복:", df["통합점포ID"].duplicated().sum())

for col in [
    "처리결과",
    "기간처리",
    "모델준비상태",
    "폐업일자_통합규칙"
]:
    if col in df.columns:
        print(f"\n[{col}]")
        display(df[col].value_counts(dropna=False).to_frame("행 수"))

print("\n[날짜 확인]")
for col in ["최초인허가일", "최근인허가일", "폐업일자_dt"]:
    dates = pd.to_datetime(df[col], errors="coerce")
    display(pd.DataFrame({
        "컬럼": [col],
        "날짜 있는 행": [dates.notna().sum()],
        "최초 날짜": [dates.min()],
        "마지막 날짜": [dates.max()]
    }))

In [ ]:
import pandas as pd

# 1. 확인할 변수
resident_cols = ["resident_yoy"]

district_cols = [
    "외지인_일평균",
    "외국인_일평균",
    "외지인_전월대비_변화율",
    "외국인_전월대비_변화율",
    "외지인_직전3개월평균대비_변화율",
    "외국인_직전3개월평균대비_변화율",
    "관광체류강도"
]

feature_cols = resident_cols + district_cols

required_cols = [
    "예측기준월", "행정동코드", "시군구코드"
] + feature_cols

missing_cols = [
    col for col in required_cols
    if col not in test_with_features.columns
]

if missing_cols:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing_cols}")

# 2. 연결된 점포 표의 결측 확인
print(f"연결된 점포 표: {len(test_with_features):,}행")
print(
    "예측기준월:",
    sorted(test_with_features["예측기준월"].astype(str).unique())
)

print("\n[① 점포 표의 지역 변수 결측]")
display(pd.DataFrame({
    "변수": feature_cols,
    "결측 행 수": [
        test_with_features[col].isna().sum()
        for col in feature_cols
    ]
}))

# 3. 같은 지역 값의 반복을 제거한 뒤 분포 확인
resident_check = test_with_features[
    ["예측기준월", "행정동코드"] + resident_cols
].drop_duplicates()

district_check = test_with_features[
    ["예측기준월", "시군구코드"] + district_cols
].drop_duplicates()

# 같은 지역·월에 서로 다른 값이 붙었는지 확인
assert not resident_check.duplicated(
    ["예측기준월", "행정동코드"]
).any(), "같은 행정동·월에 서로 다른 거주인구 값이 있습니다."

assert not district_check.duplicated(
    ["예측기준월", "시군구코드"]
).any(), "같은 구군·월에 서로 다른 방문자·관광 값이 있습니다."

print(f"\n[② 거주인구 분포: 지역·월 {len(resident_check):,}개]")
display(
    resident_check[resident_cols]
    .apply(pd.to_numeric, errors="raise")
    .describe(percentiles=[0.01, 0.5, 0.99])
    .T
)

print(f"\n[③ 방문자·관광 분포: 지역·월 {len(district_check):,}개]")
display(
    district_check[district_cols]
    .apply(pd.to_numeric, errors="raise")
    .describe(percentiles=[0.01, 0.5, 0.99])
    .T
)

print("\n완료: 기존 데이터는 변경하지 않았습니다.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
import pandas as pd

stamp = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y%m%d_%H%M%S")
folder = Path(f"/content/지역변수_연결검수_{stamp}")
folder.mkdir()

# 1. 연결된 시험용 점포 표
test_with_features.to_csv(
    folder / "202602_점포_지역변수연결_초안.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. 변수별 결측 검수표
feature_cols = [
    "resident_yoy",
    "외지인_일평균",
    "외국인_일평균",
    "외지인_전월대비_변화율",
    "외국인_전월대비_변화율",
    "외지인_직전3개월평균대비_변화율",
    "외국인_직전3개월평균대비_변화율",
    "관광체류강도"
]

pd.DataFrame({
    "변수": feature_cols,
    "결측_행수": [
        test_with_features[c].isna().sum()
        for c in feature_cols
    ]
}).to_csv(
    folder / "지역변수_결측검수.csv",
    index=False,
    encoding="utf-8-sig"
)

# 앞서 만든 연결 검수표도 함께 저장
if "join_report" in globals():
    join_report.to_csv(
        folder / "지역변수_연결검수.csv",
        index=False,
        encoding="utf-8-sig"
    )

# 3. 결과의 범위와 주의사항
(folder / "먼저읽기.md").write_text(
    """# 지역변수 연결 검수 결과

- 202602 시험용 점포 초안에 지역 변수 8개를 연결했습니다.
- 거주인구: 예측기준월 + 행정동코드로 연결했습니다.
- 방문자·관광: 예측기준월 + 시군구코드로 연결했습니다.
- 202602 예측에는 202601 관측자료를 사용했습니다.
- 실제 공개 지연을 반영하지 않은 소급평가 기준입니다.
- 인허가 최종본이 아닌 시험용 연결 검수 초안입니다.
- 학습용 점포 표 연결과 체크리스트 3-2 전체 분석은 아직 완료하지 않았습니다.
- 인허가 최종본이 확정되면 같은 연결 규칙을 적용하고 재검수해야 합니다.
""",
    encoding="utf-8"
)

# 4. ZIP 생성 및 다운로드
zip_path = folder.with_suffix(".zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for path in sorted(folder.iterdir()):
        z.write(path, arcname=path.name)

print("저장 완료:", zip_path.name)
files.download(str(zip_path))

In [ ]:
import io
import pandas as pd
from google.colab import files

# 1. 최신 점포 기본표 업로드
print("최신 인허가_master.csv를 선택해주세요.")
master_upload = files.upload()

csv_names = [
    name for name in master_upload
    if name.lower().endswith(".csv")
]
if len(csv_names) != 1:
    raise ValueError("CSV 한 개만 업로드해주세요.")

# 현재 작업에 필요한 열만 읽기
# 기존 업종 인코딩·사업장명 등은 이번 표에 복사하지 않음
use_cols = [
    "통합점포ID", "행정동코드", "구군",
    "최초인허가일", "폐업일자_dt", "영업여부"
]

master_source = pd.read_csv(
    io.BytesIO(master_upload[csv_names[0]]),
    usecols=use_cols,
    dtype="string",
    encoding="utf-8-sig"
)

master_for_snapshot = master_source.copy()

# 2. 식별자·코드 검수
for col in ["통합점포ID", "행정동코드"]:
    master_for_snapshot[col] = (
        master_for_snapshot[col].str.strip()
    )
    if (
        master_for_snapshot[col].isna().any()
        or master_for_snapshot[col].eq("").any()
    ):
        raise ValueError(f"{col}에 빈값이 있습니다.")

if master_for_snapshot["통합점포ID"].duplicated().any():
    raise ValueError("통합점포ID가 중복됩니다.")

if not master_for_snapshot["행정동코드"].str.fullmatch(
    r"\d{10}"
).all():
    raise ValueError("행정동코드가 10자리인지 확인해주세요.")

# 3. 날짜 변환 — 원본 값은 그대로 보존
for source, target in [
    ("최초인허가일", "open_s"),
    ("폐업일자_dt", "close_s")
]:
    text = master_for_snapshot[source].fillna("").str.strip()
    parsed = pd.to_datetime(
        text.mask(text.eq("")),
        format="ISO8601",
        errors="coerce"
    )
    if (text.ne("") & parsed.isna()).any():
        raise ValueError(f"{source}: 날짜 변환 실패")

    master_for_snapshot[target] = parsed

if master_for_snapshot["open_s"].isna().any():
    raise ValueError("최초인허가일이 없는 점포가 있습니다.")

if (
    master_for_snapshot["close_s"]
    < master_for_snapshot["open_s"]
).any():
    raise ValueError("폐업일이 최초인허가일보다 빠릅니다.")

state = master_for_snapshot["영업여부"].str.strip()
if not state.isin(["T", "F"]).all():
    raise ValueError("영업여부에 T·F 이외의 값이 있습니다.")

bad_status = (
    (state.eq("F") & master_for_snapshot["close_s"].isna())
    | (state.eq("T") & master_for_snapshot["close_s"].notna())
)
if bad_status.any():
    raise ValueError("영업상태와 폐업일이 맞지 않는 점포가 있습니다.")

master_for_snapshot["시군구코드"] = (
    master_for_snapshot["행정동코드"].str[:5]
)

# 4. 특정 기준일의 점포 표 생성 함수
def make_snapshot(t):
    end = t + pd.DateOffset(months=6)

    active = (
        master_for_snapshot["open_s"].lt(t)
        & (
            master_for_snapshot["close_s"].isna()
            | master_for_snapshot["close_s"].ge(t)
        )
    )

    result = master_for_snapshot.loc[
        active,
        [
            "통합점포ID", "행정동코드", "시군구코드",
            "구군", "open_s", "close_s"
        ]
    ].copy()

    result = result.rename(
        columns={"통합점포ID": "점포이력ID"}
    )
    result.insert(0, "예측기준월", t.strftime("%Y%m"))

    result["label_close_6m"] = (
        result["close_s"].ge(t)
        & result["close_s"].lt(end)
    ).astype("int8")

    return result

# 5. 학습용 월별 표 생성
# 202508의 6개월 정답은 2026년 1월 말까지
train_months = pd.date_range(
    "2023-01-01", "2025-08-01", freq="MS"
)

train_snapshot = pd.concat(
    [make_snapshot(t) for t in train_months],
    ignore_index=True
)

# 최신 master 기준 시험용 표도 별도로 생성
# 기존 test_with_features는 변경하지 않음
test_snapshot_latest = make_snapshot(
    pd.Timestamp("2026-02-01")
)

# 6. 검수·요약
assert not train_snapshot.duplicated(
    ["점포이력ID", "예측기준월"]
).any()

assert not test_snapshot_latest.duplicated(
    ["점포이력ID", "예측기준월"]
).any()

monthly_snapshot_check = (
    train_snapshot.groupby("예측기준월")
    .agg(
        점포수=("점포이력ID", "size"),
        폐업정답1=("label_close_6m", "sum"),
        폐업정답비율=("label_close_6m", "mean")
    )
    .reset_index()
)

print("최신 master 점포 수:", len(master_for_snapshot))
print("학습용 표:", train_snapshot.shape)
print("학습 기준월 수:", train_snapshot["예측기준월"].nunique())
print("최신 master 기준 시험 점포 수:", len(test_snapshot_latest))
print(
    "시험 폐업 정답 1:",
    int(test_snapshot_latest["label_close_6m"].sum())
)
print("점포·기준월 중복: 0")
print("원본 파일과 기존 연결 결과는 변경하지 않았습니다.")

display(monthly_snapshot_check)

In [ ]:
import pandas as pd

# 1. 필요한 표가 메모리에 있는지 확인
needed = [
    "train_snapshot", "test_snapshot_latest",
    "resident_features", "visitor_features", "tourism_features"
]
absent = [name for name in needed if name not in globals()]

if absent:
    raise ValueError(f"메모리에 없는 표: {absent}")

visitor_cols = [
    "외지인_일평균",
    "외국인_일평균",
    "외지인_전월대비_변화율",
    "외국인_전월대비_변화율",
    "외지인_직전3개월평균대비_변화율",
    "외국인_직전3개월평균대비_변화율"
]

all_feature_cols = (
    ["resident_yoy"] + visitor_cols + ["관광체류강도"]
)

# 2. 연결 함수
def connect_regional_features(snapshot):
    result = snapshot.copy()

    for key in ["예측기준월", "행정동코드", "시군구코드"]:
        result[key] = result[key].astype("string").str.strip()

    settings = [
        ("거주인구", resident_features, "행정동코드",
         ["resident_yoy"]),
        ("방문자", visitor_features, "시군구코드",
         visitor_cols),
        ("관광", tourism_features, "시군구코드",
         ["관광체류강도"])
    ]

    for name, source, region_key, values in settings:
        keys = ["예측기준월", region_key]
        source_cols = keys + ["관측월"] + values
        lookup = source[source_cols].copy()

        for key in keys:
            lookup[key] = lookup[key].astype("string").str.strip()

        lookup["관측월"] = (
            lookup["관측월"].astype("string").str.strip()
        )

        if lookup.duplicated(keys).any():
            raise ValueError(f"{name}: 地域・월 연결 키 중복")

        # 후보표 자체가 전월 관측자료인지 검사
        predicted = pd.to_datetime(
            lookup["예측기준월"], format="%Y%m", errors="raise"
        ).dt.to_period("M")

        expected_month = (predicted - 1).astype(str).str.replace(
            "-", "", regex=False
        )

        if not lookup["관측월"].eq(expected_month).all():
            raise ValueError(f"{name}: 전월 사용 기준 불일치")

        for col in values:
            lookup[col] = pd.to_numeric(
                lookup[col], errors="raise"
            )

        lookup = lookup.rename(
            columns={"관측월": f"{name}_관측월"}
        )

        result = result.merge(
            lookup,
            on=keys,
            how="left",
            validate="many_to_one",
            sort=False
        )

        print(f"{name} 연결 완료: {len(result):,}행")

    # 연결 과정에서 점포 목록·순서·정답이 바뀌지 않았는지 검사
    assert len(result) == len(snapshot)
    assert result["점포이력ID"].reset_index(drop=True).equals(
        snapshot["점포이력ID"].reset_index(drop=True)
    )
    assert result["label_close_6m"].reset_index(drop=True).equals(
        snapshot["label_close_6m"].reset_index(drop=True)
    )
    assert not result.duplicated(
        ["점포이력ID", "예측기준월"]
    ).any()

    return result

# 3. 학습용·최신 시험용 표 각각 연결
print("[학습용 연결]")
train_with_features = connect_regional_features(train_snapshot)

print("\n[최신 시험용 연결]")
test_with_features_latest = connect_regional_features(
    test_snapshot_latest
)

# 4. 변수별 결측 검수
feature_missing_check = pd.DataFrame({
    "변수": all_feature_cols,
    "학습용_결측행수": [
        train_with_features[c].isna().sum()
        for c in all_feature_cols
    ],
    "시험용_결측행수": [
        test_with_features_latest[c].isna().sum()
        for c in all_feature_cols
    ]
})

# 5. 월별 결측 검수
monthly_feature_missing = (
    train_with_features
    .groupby("예측기준월")[all_feature_cols]
    .count()
)

month_sizes = train_with_features.groupby("예측기준월").size()

monthly_feature_missing = (
    monthly_feature_missing
    .rsub(month_sizes, axis=0)
    .reset_index()
)

print("\n[변수별 결측]")
display(feature_missing_check)

print("\n[월별 결측]")
display(monthly_feature_missing)

print("\n학습용 연결 결과:", train_with_features.shape)
print("최신 시험용 연결 결과:", test_with_features_latest.shape)
print("행 수·점포 목록·폐업 정답 유지 확인 완료")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 图中用F1~F8标识，避免韩文字体显示问题
feature_map = {
    "F1": "resident_yoy",
    "F2": "외지인_일평균",
    "F3": "외국인_일평균",
    "F4": "외지인_전월대비_변화율",
    "F5": "외국인_전월대비_변화율",
    "F6": "외지인_직전3개월평균대비_변화율",
    "F7": "외국인_직전3개월평균대비_변화율",
    "F8": "관광체류강도"
}
feature_cols = list(feature_map.values())
short_names = list(feature_map)

print("[지표 번호 안내]")
display(pd.DataFrame(
    feature_map.items(), columns=["그래프 번호", "변수"]
))

# 1. 合并后点铺表的基础统计
combined_stats = (
    train_with_features[feature_cols]
    .describe(percentiles=[0.01, 0.5, 0.99])
    .T
)
combined_stats["결측률"] = (
    train_with_features[feature_cols].isna().mean()
)

print("\n[① 학습용 점포 표 기초통계]")
display(combined_stats)

# 2. 数值范围检查：变化率可超过100%，但不能低于-100%
range_rows = []
for col in feature_cols:
    x = train_with_features[col]

    if col in [
        "외지인_일평균", "외국인_일평균", "관광체류강도"
    ]:
        bad = x.lt(0)
        rule = "음수 여부"
    else:
        bad = x.lt(-1)
        rule = "변화율 -100% 미만 여부"

    range_rows.append({
        "변수": col,
        "검사기준": rule,
        "범위위반행수": int(bad.sum()),
        "무한대행수": int(np.isinf(x).sum())
    })

range_check = pd.DataFrame(range_rows)
print("\n[② 값 범위 점검]")
display(range_check)

# 3. Histograms: count each regional value once per month
# Resident: dong-month; visitors/tourism: district-month
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for ax, (short, col) in zip(axes.flat, feature_map.items()):
    region = "행정동코드" if col == "resident_yoy" else "시군구코드"
    values = (
        train_with_features[
            ["예측기준월", region, col]
        ]
        .drop_duplicates()[col]
        .dropna()
    )
    ax.hist(values, bins=30, edgecolor="white")
    ax.set_title(short)
    ax.set_ylabel("Region-month count")

plt.suptitle("Training feature distributions")
plt.tight_layout()
plt.show()

# 4. Feature correlations in the actual shop-month table
# Regions with more shops receive more weight through repeated rows.
# This is descriptive correlation, not an independence test.
feature_corr = train_with_features[feature_cols].corr()

high_corr_rows = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = feature_corr.iloc[i, j]
        if pd.notna(r) and abs(r) > 0.8:
            high_corr_rows.append({
                "변수1": feature_cols[i],
                "변수2": feature_cols[j],
                "상관계수": r
            })

high_corr_pairs = pd.DataFrame(
    high_corr_rows,
    columns=["변수1", "변수2", "상관계수"]
)

print("\n[③ 상관계수 절댓값 0.8 초과 쌍]")
display(high_corr_pairs)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(feature_corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(8), short_names)
ax.set_yticks(range(8), short_names)

for i in range(8):
    for j in range(8):
        r = feature_corr.iloc[i, j]
        ax.text(
            j, i, f"{r:.2f}" if pd.notna(r) else "NA",
            ha="center", va="center", fontsize=8
        )

fig.colorbar(im, ax=ax)
ax.set_title("Training shop-month feature correlations")
plt.tight_layout()
plt.show()

# 5. Monthly shop counts and six-month closure-label rate
monthly_eda = (
    train_with_features.groupby("예측기준월")
    .agg(
        점포수=("점포이력ID", "size"),
        폐업정답1=("label_close_6m", "sum"),
        폐업정답비율=("label_close_6m", "mean")
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
months = monthly_eda["예측기준월"].astype(str)

axes[0].plot(months, monthly_eda["점포수"])
axes[0].set_ylabel("Active shops")

axes[1].plot(
    months, monthly_eda["폐업정답비율"] * 100,
    color="darkorange"
)
axes[1].set_ylabel("6-month closure label (%)")
axes[1].set_xticks(range(0, len(months), 4))
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# 6. District comparison — training only
district_summary = (
    train_with_features.groupby("구군")
    .agg(
        점포월행수=("점포이력ID", "size"),
        폐업정답1=("label_close_6m", "sum"),
        폐업정답비율=("label_close_6m", "mean")
    )
)

district_means = (
    train_with_features.groupby("구군")[feature_cols].mean()
)

district_eda = (
    district_summary.join(district_means)
    .sort_values("폐업정답비율", ascending=False)
    .reset_index()
)

print("\n[④ 구군별 폐업 정답 비율·지표 평균 — 학습 구간]")
display(district_eda)

print("""
분석 완료.
- 변화율 0.1은 10%이며, 100% 초과 자체는 오류가 아닙니다.
- 월별 폐업정답비율은 해당 기준월 이후 6개월의 비율입니다.
- 같은 점포가 여러 달에 반복되므로 점포월행수는 고유 점포 수가 아닙니다.
- 상관이 높다고 자동으로 변수를 제거하지 않습니다.
- 업종별 비교는 과거 기준월의 업종 정보 확인 후 별도로 진행합니다.
""")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 학습용 정답에 포함된 폐업 점포
# 여러 기준월에 반복된 같은 점포는 한 번만 집계
closure_events = (
    train_with_features.loc[
        train_with_features["label_close_6m"].eq(1),
        ["점포이력ID", "구군", "close_s"]
    ]
    .drop_duplicates()
    .copy()
)

# 같은 점포에 폐업일이 여러 개 있으면 확인 필요
if closure_events["점포이력ID"].duplicated().any():
    raise ValueError("같은 점포에 서로 다른 구군·폐업일이 있습니다.")

closure_events["폐업일"] = (
    pd.to_datetime(closure_events["close_s"]).dt.normalize()
)
closure_events["폐업월"] = (
    closure_events["폐업일"].dt.strftime("%Y%m")
)

# 2. 실제 폐업월별 고유 점포 수
closure_monthly = (
    closure_events.groupby("폐업월")
    .size()
    .rename("폐업점포수")
    .reset_index()
)

# 3. 실제 폐업일별 고유 점포 수
closure_daily = (
    closure_events.groupby(["폐업월", "폐업일"])
    .size()
    .rename("폐업점포수")
    .reset_index()
)

closure_daily = closure_daily.merge(
    closure_monthly.rename(
        columns={"폐업점포수": "해당월_폐업점포수"}
    ),
    on="폐업월",
    how="left",
    validate="many_to_one"
)

closure_daily["해당월_내_비중"] = (
    closure_daily["폐업점포수"]
    / closure_daily["해당월_폐업점포수"]
)

closure_daily_top = (
    closure_daily.sort_values(
        "폐업점포수", ascending=False
    ).head(20)
)

print("학습용 정답에 포함된 고유 폐업 점포 수:",
      f"{len(closure_events):,}")
print("※ 실제 폐업일 집계이며, 월별 6개월 정답 합계와 다릅니다.")

print("\n[① 실제 폐업월별 점포 수]")
display(closure_monthly)

print("\n[② 폐업 기록이 많은 날짜 상위 20개]")
display(closure_daily_top)

# 4. 월별 실제 폐업 기록 그래프
fig, ax = plt.subplots(figsize=(12, 4))
positions = range(len(closure_monthly))

ax.bar(positions, closure_monthly["폐업점포수"])
ticks = list(range(0, len(closure_monthly), 3))
ax.set_xticks(
    ticks,
    closure_monthly["폐업월"].iloc[ticks],
    rotation=45
)
ax.set_ylabel("Unique closed shops")
ax.set_title("Closure events used in training labels")
plt.tight_layout()
plt.show()

print("""
점검 완료.
특정 날짜에 기록이 많다는 사실만으로 오류나 일괄 행정처리로
확정하지 않습니다. 원자료 확인이 필요한 날짜를 찾는 용도입니다.
기존 점포·정답·지역 변수는 변경하지 않았습니다.
""")

In [ ]:
import io
import pandas as pd
from google.colab import files

# 1. 인허가별 과거 업종 자료 읽기
print("인허가_폐업정보결합.csv 또는 (1).csv를 선택해주세요.")
industry_upload = files.upload()

names = [
    n for n in industry_upload
    if n.lower().endswith(".csv")
]
if len(names) != 1:
    raise ValueError("CSV 한 개만 선택해주세요.")

permit_industry = pd.read_csv(
    io.BytesIO(industry_upload[names[0]]),
    usecols=[
        "점포ID", "점포이력ID", "업종명", "인허가일자_dt"
    ],
    dtype="string",
    encoding="utf-8-sig"
)

for c in permit_industry.columns:
    permit_industry[c] = permit_industry[c].str.strip()

if permit_industry[
    ["점포ID", "점포이력ID", "업종명"]
].isna().any().any():
    raise ValueError("식별자 또는 업종명에 빈값이 있습니다.")

if permit_industry["점포ID"].duplicated().any():
    raise ValueError("인허가별 원본이 아닐 수 있습니다. 점포ID 중복 확인.")

permit_industry["_등록일"] = pd.to_datetime(
    permit_industry["인허가일자_dt"],
    format="ISO8601",
    errors="raise"
)

if permit_industry["_등록일"].isna().any():
    raise ValueError("등록일이 없는 기록이 있습니다.")

feature_cols = [
    "resident_yoy",
    "외지인_일평균",
    "외국인_일평균",
    "외지인_전월대비_변화율",
    "외국인_전월대비_변화율",
    "외지인_직전3개월평균대비_변화율",
    "외국인_직전3개월평균대비_변화율",
    "관광체류강도"
]

# 2. 월별로 업종 연결 후 집계
# 메모리를 줄이기 위해 전체 404만 행을 다시 합치지 않음
industry_parts = []
industry_audit_rows = []

for month, part in train_with_features.groupby(
    "예측기준월", sort=True
):
    t = pd.to_datetime(str(month), format="%Y%m")

    eligible = permit_industry.loc[
        permit_industry["_등록일"].lt(t)
        & permit_industry["점포이력ID"].isin(part["점포이력ID"])
    ].copy()

    latest_dates = eligible.groupby(
        "점포이력ID"
    )["_등록일"].transform("max")

    latest = eligible.loc[
        eligible["_등록일"].eq(latest_dates)
    ]

    lookup = (
        latest.groupby("점포이력ID")
        .agg(
            대표업종=("업종명", "first"),
            최근일_업종수=("업종명", "nunique")
        )
        .reset_index()
    )

    # 같은 최근 날짜에 업종이 여러 개면 임의 선택하지 않음
    lookup.loc[
        lookup["최근일_업종수"].gt(1), "대표업종"
    ] = "동일일복수업종_선택보류"

    joined = part[
        ["점포이력ID", "label_close_6m"] + feature_cols
    ].merge(
        lookup,
        on="점포이력ID",
        how="left",
        validate="one_to_one",
        sort=False
    )

    assert len(joined) == len(part)

    not_found = joined["대표업종"].isna()
    joined["대표업종"] = joined["대표업종"].fillna(
        "원자료대응없음_확인필요"
    )

    industry_audit_rows.append({
        "예측기준월": month,
        "점포수": len(joined),
        "업종대응없음": int(not_found.sum()),
        "동일일복수업종_선택보류": int(
            joined["대표업종"]
            .eq("동일일복수업종_선택보류").sum()
        )
    })

    # 월별 평균을 단순 평균하지 않도록 합계와 유효 개수 보관
    aggregations = {
        "점포월행수": ("점포이력ID", "size"),
        "폐업정답1": ("label_close_6m", "sum")
    }

    for i, col in enumerate(feature_cols):
        aggregations[f"sum_{i}"] = (col, "sum")
        aggregations[f"count_{i}"] = (col, "count")

    summary = (
        joined.groupby("대표업종")
        .agg(**aggregations)
        .reset_index()
    )

    industry_parts.append(summary)

# 3. 전체 학습 기간 업종별 요약
industry_totals = (
    pd.concat(industry_parts, ignore_index=True)
    .groupby("대표업종")
    .sum(numeric_only=True)
)

industry_eda = industry_totals[
    ["점포월행수", "폐업정답1"]
].copy()

industry_eda["폐업정답비율"] = (
    industry_eda["폐업정답1"]
    / industry_eda["점포월행수"]
)

for i, col in enumerate(feature_cols):
    denominator = industry_totals[f"count_{i}"].replace(0, float("nan"))
    industry_eda[col] = (
        industry_totals[f"sum_{i}"] / denominator
    )

industry_eda = (
    industry_eda.reset_index()
    .sort_values("점포월행수", ascending=False)
)

industry_link_audit = pd.DataFrame(industry_audit_rows)

assert industry_eda["점포월행수"].sum() == len(train_with_features)

print("[① 월별 업종 연결 검수]")
display(industry_link_audit)

print("\n[② 업종별 비교 — 점포월행수 상위 20개]")
display(industry_eda.head(20))

print("\n전체 업종별 결과 행 수:", len(industry_eda))
print("전체 학습 점포월행수:", industry_eda["점포월행수"].sum())
print("기존 학습표·시험표·정답은 변경하지 않았습니다.")
print("※ 대표업종은 기준월 이전 최근 인허가 기준의 EDA용 분류입니다.")
print("※ 복수업종 보류 집단은 실제 단일 업종으로 해석하지 않습니다.")

In [ ]:
from pathlib import Path
import pandas as pd

# 1. 필요한 최신 결과 확인
required_names = [
    "resident_features", "visitor_features", "tourism_features",
    "train_with_features", "test_with_features_latest",
    "feature_missing_check", "monthly_feature_missing",
    "combined_stats", "range_check", "feature_corr",
    "high_corr_pairs", "monthly_eda", "district_eda",
    "closure_monthly", "closure_daily_top"
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise ValueError(
        "아직 메모리에 없는 결과가 있습니다. 런타임을 재시작하지 마세요.\n"
        + "\n".join(missing)
    )

# 최신 표가 맞는지 확인
assert set(
    train_with_features["예측기준월"].astype(str)
) == set(pd.date_range(
    "2023-01-01", "2025-08-01", freq="MS"
).strftime("%Y%m"))

assert set(
    test_with_features_latest["예측기준월"].astype(str)
) == {"202602"}

# 2. 새 전달 폴더 생성 — 기존 폴더 덮어쓰기 금지
stamp = pd.Timestamp.now(
    tz="Asia/Seoul"
).strftime("%Y%m%d_%H%M%S")

delivery_dir = Path(
    f"/content/이채우_지역변수_EDA_최종전달_{stamp}"
)
variable_dir = delivery_dir / "모델연결용"
audit_dir = delivery_dir / "검수_요약표"

variable_dir.mkdir(parents=True, exist_ok=False)
audit_dir.mkdir()

# 3. 최신 모델 연결용 CSV 저장·키 검수
variable_settings = [
    ("방문자", visitor_features, "시군구코드"),
    ("거주인구", resident_features, "행정동코드"),
    ("관광체류강도", tourism_features, "시군구코드")
]

variable_inventory = []

for name, frame, region_key in variable_settings:
    out = frame.copy()

    keys = ["예측기준월", region_key]
    for key in keys:
        out[key] = out[key].astype("string").str.strip()
        if out[key].isna().any() or out[key].eq("").any():
            raise ValueError(f"{name}: 연결 키에 빈값이 있습니다.")

    if out.duplicated(keys).any():
        raise ValueError(f"{name}: 연결 키가 중복됩니다.")

    # 모두 전월 관측 기준인지 확인
    prediction = pd.to_datetime(
        out["예측기준월"], format="%Y%m"
    ).dt.to_period("M")

    expected_observation = (
        (prediction - 1).astype(str)
        .str.replace("-", "", regex=False)
    )

    if not out["관측월"].astype(str).eq(
        expected_observation
    ).all():
        raise ValueError(f"{name}: 전월 관측 기준이 아닙니다.")

    # 거주인구 전년 비교월도 확인
    if name == "거주인구":
        expected_year_ago = (
            (prediction - 13).astype(str)
            .str.replace("-", "", regex=False)
        )
        if not out["전년비교월"].astype(str).eq(
            expected_year_ago
        ).all():
            raise ValueError("거주인구 전년 비교월이 맞지 않습니다.")

    path = variable_dir / f"{name}_변수.csv"
    out.to_csv(path, index=False, encoding="utf-8-sig")

    # 저장한 CSV를 다시 읽어 코드·행 수 확인
    check = pd.read_csv(path, dtype="string", encoding="utf-8-sig")
    assert len(check) == len(out)
    for key in keys:
        assert check[key].equals(out[key].reset_index(drop=True))

    variable_inventory.append({
        "파일": path.name,
        "행수": len(out),
        "열수": len(out.columns),
        "시작_예측기준월": out["예측기준월"].min(),
        "종료_예측기준월": out["예측기준월"].max(),
        "연결키": " + ".join(keys)
    })

# 4. 실제 실행한 검수·집계 결과 저장
audit_tables = {
    "변수별_기초통계.csv": combined_stats,
    "변수별_결측검수.csv": feature_missing_check,
    "월별_결측검수.csv": monthly_feature_missing,
    "값범위_검수.csv": range_check,
    "지표_상관행렬.csv": feature_corr,
    "고상관_변수쌍.csv": high_corr_pairs,
    "월별_점포수_폐업비율.csv": monthly_eda,
    "구군별_비교.csv": district_eda,
    "폐업월별_점포수.csv": closure_monthly,
    "폐업일_집중검토.csv": closure_daily_top
}

# 두 표는 행 인덱스에 변수명이 있으므로 함께 저장
index_tables = {
    "변수별_기초통계.csv",
    "지표_상관행렬.csv"
}

for filename, frame in audit_tables.items():
    frame.to_csv(
        audit_dir / filename,
        index=filename in index_tables,
        encoding="utf-8-sig"
    )

# 5. 완료 범위 명시
checklist_status = pd.DataFrame([
    ["기초통계·분포·극단값", "담당 변수 8개 확인 완료"],
    ["결측률·시작 월", "확인 완료"],
    ["월별 점포 수·폐업 정답 비율", "확인 완료"],
    ["폐업 날짜 집중", "확인 완료·집중 원인 미확인"],
    ["구군별 폐업 비율·지표 평균", "확인 완료"],
    ["업종별 폐업 비율·지표 평균", "미진행"],
    ["상관행렬·절댓값 0.8 초과 쌍", "확인 완료"],
    ["값 범위·무한대", "적용한 검사 기준 위반 0"],
    ["팀 공유", "전달물 작성 중"]
], columns=["체크리스트_3_2_항목", "상태"])

checklist_status.to_csv(
    audit_dir / "체크리스트_진행상태.csv",
    index=False,
    encoding="utf-8-sig"
)

inventory = pd.DataFrame(variable_inventory)
inventory.to_csv(
    audit_dir / "모델연결용_파일목록.csv",
    index=False,
    encoding="utf-8-sig"
)

# 6. 연결 기준과 해석 주의사항
readme = f"""# 세 지역 자료 EDA 전달 안내

## 범위
방문자·거주인구·관광체류강도의 지역 변수 8개를 대상으로
개별 EDA, 점포 표 연결, 통합 EDA를 수행했다.
카드·통신 등 다른 담당자의 변수는 분석 범위에 포함되지 않는다.
업종별 비교는 미진행이며 체크리스트 3-2 전체 완료로 표시하지 않는다.

## 다음 담당자가 사용할 파일
모델연결용/ 아래 CSV 3개를 최종 점포 학습표에 LEFT JOIN한다.
CSV를 읽을 때 지역 코드와 예측기준월은 문자열로 읽는다.
기존 점포 행을 삭제하거나 동일 키의 여러 값을 연결하지 않는다.

- 거주인구: 예측기준월 + 행정동코드
- 방문자·관광체류강도: 예측기준월 + 시군구코드
- 지역 변수 연결은 many-to-one 관계여야 한다.
- 연결 전후 점포 행 수와 점포·기준월 중복을 점검한다.

## 변수 정의
- resident_yoy:
  전월 거주인구 / 전년도 같은 관측월 거주인구 - 1
- 외지인·외국인 일평균:
  해당 관측월의 일평균 방문자 수. 주민등록 인구가 아니다.
- 전월 대비 변화율:
  관측월 일평균 / 직전월 일평균 - 1
- 직전 3개월 평균 대비 변화율:
  관측월 일평균 / 관측월을 제외한 직전 3개월 일평균 값의 평균 - 1
- 관광체류강도:
  원자료 지표 코드 21의 지수 값. 백분율이나 방문자 수가 아니다.
- 변화율 0.1은 10%이며, 100%를 초과할 수 있다.
- 행정동 변수와 구군 변수의 공간 단위가 다르다.
  구군 값을 행정동 자체의 관측값으로 해석하지 않는다.

## 시점 기준
모든 변수는 예측기준월의 전월 관측자료를 사용한다.
예: 202602 예측에는 202601 관측자료 사용.
실제 공개 지연과 과거 데이터 개정은 반영하지 않은 소급평가다.

## 이번 통합 EDA의 점포 표
- 최신 인허가_master.csv 기준
- 학습 기준월: 202301~202508
- 학습 점포·월 행 수: {len(train_with_features):,}
- 시험 기준월: 202602
- 시험 점포 수: {len(test_with_features_latest):,}
- 시험 정답 기간: 2026-02-01 이상~2026-08-01 미만
- 202508 학습 기준월의 정답은 2026년 1월까지 확인한다.
- 학습 기간은 팀 최종 명세와 맞춰 사용해야 한다.
- 통합 master의 최초 인허가일과 폐업일을 사용한 분석용 표다.
  최초 인허가일이 실제 개업일과 같다는 의미는 아니다.
- 원본 날짜와 영업상태는 모델 입력에서 제외한다.
  기준일 이전 정보로 계산한 업력 등은 별도 설계한다.

## 주요 발견
- 202301 방문자·관광 결측은 202212 관측자료 부재 때문이다.
  결측을 0으로 바꾸거나 점포를 임의로 삭제하지 않았다.
- 방문자 변화율의 파일 내부 재계산 가능 구간은 검산 결과 일치했다.
  파일 이전 과거자료가 필요한 초기 변화율은 직접 재검산하지 못했다.
- 거주인구의 신호동 인구는 녹산동에 포함된다는 담당자 확인을 적용했다.
- 전월 대비와 직전 3개월 평균 대비 변화율 간 고상관 쌍 2개를 표시했다.
  자동 변수 삭제는 하지 않았다.
- 점포·월 통계에는 같은 점포와 같은 지역 값이 반복된다.
  합계는 고유 점포 수가 아니며, 상관계수는 독립성 검정이 아니다.
- 히스토그램은 지역·월별 중복을 제거했고,
  기초통계·상관·구군 평균은 점포·월 기준으로 계산했다.
- 구군별 폐업 비율 차이를 인과관계나 향후 예측 성능으로 해석하지 않는다.
- 특정 폐업일과 연말에 기록이 집중된다.
  오류 또는 일괄 행정처리로 확정하지 않았으며 기록을 유지했다.

## 남은 사항
- 과거 기준월의 업종 기준을 확정한 후 업종별 비교
- 폐업 날짜 집중 원인 확인
- 실제 운영에 적용할 경우 자료 공개 시점 검토
- 이번 전달물은 완성된 전체 모델 학습표가 아니다.
"""

(delivery_dir / "00_먼저읽기.md").write_text(
    readme, encoding="utf-8"
)

print("\n전달용 CSV·설명서 저장 완료")
print("저장 위치:", delivery_dir)
display(inventory)
display(checklist_status)

print(
    "\n아직 최종 ZIP은 만들지 않았습니다. "
    "노트북·핵심요약 PDF를 추가한 뒤 한 번만 압축합니다."
)

In [ ]:
import shutil
from google.colab import files

# 방금 생성한 delivery_dir 폴더만 압축
archive = shutil.make_archive(
    "/content/EDA_최종정리_작업자료",
    "zip",
    root_dir=str(delivery_dir)
)

files.download(archive)